# LPAC et portée de communication : le modèle publié se dégrade-t-il au-delà de 512 m ?

Projet CS 224W (Stanford, automne 2026). Ce notebook reproduit, **sans aucun entraînement**,
l'expérience de la section VI-H de *LPAC* (Agarwal et al., IEEE T-RO 2025, arXiv:2401.04855)
avec le modèle pré-entraîné publié dans la release v1.3.1 de
[KumarRobotics/CoverageControl](https://github.com/KumarRobotics/CoverageControl).

Il s'exécute de haut en bas (`Runtime → Run all`) et écrit ses résultats dans `OUT_DIR`
(par défaut `/content/lpac_results`, ou sur Google Drive si `USE_DRIVE = True`) :

| fichier | contenu |
|---|---|
| `results.csv` | une ligne par (environnement, portée, nombre de robots, contrôleur) : coût normalisé, degré moyen |
| `rank.csv` | rang effectif en sortie de chaque couche TAGConv (LPAC) |
| `curves.csv` | courbes de coût sous-échantillonnées |
| `fig_portee.png`, `fig_degre.png`, `fig_degre_constant.png`, `fig_rang.png` | figures |
| `lpac_results.zip` | le tout, à télécharger |

**Durée.** Les baselines CVT tournent en C++ sur CPU (Colab n'en a que 2) et LPAC sur GPU.
Comptez 3 à 5 h pour le balayage complet. Le balayage est **reprenable** : relancer le
notebook saute les lignes déjà présentes dans `results.csv` (d'où l'intérêt de Drive).
Pour un test rapide, mettez `QUICK = True` (1 environnement, 60 pas, ~2 min).

**Mise en garde.** Un seul modèle est publié, entraîné à 128 m. L'évaluer à 1024 m mélange
l'effet de la densité du graphe et l'écart au régime d'entraînement ; le balayage à degré
constant sert à les séparer, au prix d'un autre défaut (le nombre de robots change la tâche).

In [ ]:
# Dépendances : le simulateur (roue manylinux, Python 3.10 à 3.13) et PyTorch Geometric.
# torch et torchvision sont déjà présents sur Colab.
import importlib, subprocess, sys
for pkg, spec in [("coverage_control", "coverage_control==1.3.1"), ("torch_geometric", "torch_geometric")]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", spec])
import torch, torch_geometric, coverage_control
print("torch", torch.__version__, "| PyG", torch_geometric.__version__,
      "| coverage_control", coverage_control.__version__, "| cuda", torch.cuda.is_available())

In [ ]:
# Réglages du run. Tout est public et déterministe (environnements et modèle de la release).
import os
from pathlib import Path
QUICK = os.environ.get("LPAC_QUICK", "0") == "1"   # True : 1 environnement, 60 pas, pour vérifier que tout tourne
USE_DRIVE = False      # True : résultats persistants dans Google Drive (reprise après déconnexion)
ROOT = Path(os.environ.get("LPAC_NB_ROOT", "/content"))  # racine de travail (Colab : /content)
NUM_ENVS = 11          # environnements d'évaluation disponibles dans la release
NUM_STEPS = 600        # NumSteps de eval.toml
RANGES = [128, 256, 512, 768, 1024]                      # portées (m), 32 robots
CONSTANT_DEGREE = ["256:32", "512:11", "768:7", "1024:6"]  # (portée, robots) à degré ~constant

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/lpac_results")
else:
    OUT_DIR = ROOT / "lpac_results"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if QUICK:
    NUM_ENVS, NUM_STEPS, RANGES, CONSTANT_DEGREE = 1, 60, [128, 1024], ["1024:6"]
print("OUT_DIR =", OUT_DIR)

In [ ]:
# Modèle pré-entraîné, 11 environnements d'évaluation et paramètres des auteurs.
import os, tarfile, urllib.request
DATA_ROOT = ROOT / "lpac_data"
DATA_DIR = DATA_ROOT / "lpac"
if not (DATA_DIR / "models" / "model_k3_1024_state_dict.pt").is_file():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    archive = DATA_ROOT / "lpac_CoverageControl.tar.gz"
    urllib.request.urlretrieve("https://github.com/KumarRobotics/CoverageControl/releases/download/v1.3.1/lpac_CoverageControl.tar.gz", archive)
    with tarfile.open(archive) as tar:
        tar.extractall(DATA_ROOT)
os.environ["LPAC_DATA_DIR"] = str(DATA_DIR)
print(sorted(p.name for p in DATA_DIR.iterdir()), "|", len(list((DATA_DIR / "envs").glob("*.env"))), "environnements")

## Le code : package `lpac_study`

Les cellules suivantes écrivent le package sur le disque du runtime. Il est identique à celui
du dépôt du projet. Tout ce qui touche au graphe passe par PyTorch Geometric :
`CoverageEnvUtils.get_torch_geometric_data` construit l'objet `Data` (graphe à rayon,
symétrique), `torch_geometric.utils.degree` mesure le degré, et le GNN est une pile de
cinq `torch_geometric.nn.TAGConv` (K = 3). Note de lecture : `LPAC.forward` ne transmet
pas `edge_weight` au GNN, l'adjacence vue par TAGConv est donc binaire, normalisée
symétriquement, sans boucles propres.

In [ ]:
import os
os.makedirs('lpac_study', exist_ok=True)

In [ ]:
%%writefile lpac_study/__init__.py
"""Reproduction study of the LPAC communication-range anomaly (Agarwal et al., T-RO 2025).

The package evaluates the published, pre-trained LPAC-K3 model and the CVT
baselines of the CoverageControl library under varying communication ranges,
and records graph statistics (degree, effective rank of GNN layer outputs)
along the LPAC trajectories.  No training is performed anywhere.
"""

In [ ]:
%%writefile lpac_study/paths.py
"""Locate the files of the published LPAC release archive without hard-coded paths."""
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

DATA_DIR_ENV_VAR = "LPAC_DATA_DIR"
RELEASE_URL = (
    "https://github.com/KumarRobotics/CoverageControl/releases/download/"
    "v1.3.1/lpac_CoverageControl.tar.gz"
)


@dataclass(frozen=True)
class DataPaths:
    """Paths inside the extracted ``lpac/`` folder of the release archive."""

    root: Path

    @property
    def model_state_dict(self) -> Path:
        return self.root / "models" / "model_k3_1024_state_dict.pt"

    @property
    def learning_params(self) -> Path:
        return self.root / "params" / "learning_params.toml"

    @property
    def env_params(self) -> Path:
        return self.root / "params" / "coverage_control_params.toml"

    @property
    def envs_dir(self) -> Path:
        return self.root / "envs"

    def env_file(self, index: int) -> Path:
        """Importance density field (Gaussian features) of environment ``index``."""
        return self.envs_dir / f"{index}.env"

    def pos_file(self, index: int) -> Path:
        """Initial robot positions of environment ``index`` (one ``x y`` pair per line)."""
        return self.envs_dir / f"{index}.pos"

    def count_environments(self) -> int:
        """Number of environments for which both ``.env`` and ``.pos`` files exist."""
        count = 0
        while self.env_file(count).is_file() and self.pos_file(count).is_file():
            count += 1
        return count

    def check(self) -> None:
        """Raise a clear error when a required file is missing."""
        for path in (self.model_state_dict, self.learning_params, self.env_params):
            if not path.is_file():
                raise FileNotFoundError(f"Missing release file: {path}")
        if self.count_environments() == 0:
            raise FileNotFoundError(f"No environment files found in {self.envs_dir}")


def resolve_data_dir(cli_value: str | None = None) -> DataPaths:
    """Resolve the data directory from the CLI, else from ``$LPAC_DATA_DIR``.

    The directory is the ``lpac/`` folder produced by extracting the release
    archive (it contains ``models/``, ``envs/`` and ``params/``).
    """
    value = cli_value or os.environ.get(DATA_DIR_ENV_VAR)
    if not value:
        raise SystemExit(
            f"Give --data-dir or set {DATA_DIR_ENV_VAR} to the extracted 'lpac' folder."
        )
    paths = DataPaths(Path(value).expanduser().resolve())
    paths.check()
    return paths

In [ ]:
%%writefile lpac_study/simulation.py
"""Thin wrappers around the CoverageControl simulator shared by all experiments.

Everything here is deterministic: the simulator only draws random numbers when
it generates a new environment or when position noise is enabled, and both
are avoided (environments come from the release archive, noise is off).
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import coverage_control as cc
from coverage_control import CoverageSystem, PointVector, WorldIDF


def make_parameters(
    env_params_file: Path, communication_range: float, num_robots: int
) -> cc.Parameters:
    """Load the authors' simulation parameters and override range and team size.

    ``CoverageSystem`` copies the parameters at construction, so the overrides
    must happen before the environment is built.
    """
    params = cc.Parameters(str(env_params_file))
    params.pCommunicationRange = float(communication_range)
    params.pNumRobots = int(num_robots)
    return params


def load_initial_positions(pos_file: Path, num_robots: int) -> np.ndarray:
    """Return the first ``num_robots`` initial positions stored in a ``.pos`` file.

    The released files hold 32 positions.  The simulator refuses a ``.pos``
    file whose length differs from ``NumRobots``, so smaller teams are built by
    keeping the first rows of the file (a fixed, documented, seed-free choice).
    """
    positions = np.loadtxt(pos_file, dtype=np.float64).reshape(-1, 2)
    if num_robots > len(positions):
        raise ValueError(
            f"{pos_file} holds {len(positions)} positions, fewer than {num_robots}"
        )
    return positions[:num_robots]


def make_environment(
    params: cc.Parameters, env_file: Path, initial_positions: np.ndarray
) -> CoverageSystem:
    """Build a fresh coverage environment from an IDF file and initial positions."""
    world_idf = WorldIDF(params, str(env_file))
    return CoverageSystem(params, world_idf, PointVector(initial_positions))


def normalized_cost(env: CoverageSystem, initial_cost: float) -> float:
    """Coverage cost divided by the cost at the start of the episode (as in the paper)."""
    return env.GetObjectiveValue() / initial_cost

In [ ]:
%%writefile lpac_study/graph_stats.py
"""Statistics of the communication graph actually fed to the GNN."""
from __future__ import annotations

from dataclasses import dataclass

import torch
from torch_geometric.utils import degree


@dataclass(frozen=True)
class DegreeStats:
    """Per-step degree statistics of one communication graph."""

    mean: float
    std: float
    isolated_fraction: float


def degree_stats(edge_index: torch.Tensor, num_nodes: int) -> DegreeStats:
    """Mean and standard deviation of node degree from a PyG ``edge_index``.

    The communication graph is symmetric, so in- and out-degree coincide.
    """
    deg = degree(edge_index[1], num_nodes=num_nodes, dtype=torch.float32)
    return DegreeStats(
        mean=float(deg.mean()),
        std=float(deg.std(unbiased=False)),
        isolated_fraction=float((deg == 0).float().mean()),
    )

In [ ]:
%%writefile lpac_study/effective_rank.py
"""Effective rank of a node-representation matrix (Roy & Vetterli, 2007).

Used as the over-smoothing measure recommended by Zhang et al. (ICLR 2026,
arXiv:2502.04591): it collapses towards 1 when all node representations
become collinear, where the Dirichlet energy stays uninformative for shallow
networks.
"""
from __future__ import annotations

import math

import torch


def effective_rank(features: torch.Tensor) -> float:
    """Exponential of the Shannon entropy of the normalized singular values.

    Args:
        features: matrix of shape ``(num_nodes, num_channels)``.

    Returns:
        A value in ``[1, min(num_nodes, num_channels)]``, or ``nan`` when the
        matrix is identically zero.
    """
    singular_values = torch.linalg.svdvals(features.detach().double())
    total = singular_values.sum()
    if total <= 0:
        return math.nan
    probabilities = singular_values / total
    probabilities = probabilities[probabilities > 0]
    entropy = -(probabilities * probabilities.log()).sum()
    return float(torch.exp(entropy))

In [ ]:
%%writefile lpac_study/lpac_policy.py
"""The published LPAC-K3 policy, instrumented to expose its communication graph.

The wrapper mirrors ``coverage_control.algorithms.ControllerNN`` (v1.3.1) but
keeps the PyG ``Data`` object fed to the network and captures the output of
each of the five ``TAGConv`` layers through forward hooks.  ``torch.compile``
is not used so that hooks and CPU execution stay simple and deterministic.
"""
from __future__ import annotations

from pathlib import Path

import torch
import coverage_control as cc
import coverage_control.nn as cc_nn
from coverage_control import CoverageEnvUtils, CoverageSystem, IOUtils, PointVector
from torch_geometric.data import Data

from .paths import DataPaths

# Controller options of the authors' eval.toml ([[Controllers]] lpac entry).
USE_CNN = True
USE_COMM_MAP = True
CNN_MAP_SIZE = 32


class LPACPolicy:
    """Pre-trained LPAC model with access to its graph and layer activations."""

    def __init__(self, paths: DataPaths, device: torch.device):
        learning_params = IOUtils.load_toml(str(paths.learning_params))
        self.device = device
        self.model = cc_nn.LPAC(learning_params).to(device)
        self.model.load_model(str(paths.model_state_dict))
        self.model.eval()
        self.actions_mean = self.model.actions_mean.to(device)
        self.actions_std = self.model.actions_std.to(device)
        self.num_layers = self.model.gnn_backbone.num_layers
        self._layer_outputs: list[torch.Tensor] = []
        self._register_layer_hooks()

    def _register_layer_hooks(self) -> None:
        """Capture the post-ReLU output of every TAGConv layer of the GNN backbone."""
        backbone = self.model.gnn_backbone
        for layer_index in range(self.num_layers):
            layer = backbone._modules[f"graph_conv_{layer_index}"]
            layer.register_forward_hook(self._store_layer_output)

    def _store_layer_output(self, module, inputs, output) -> None:
        # GNNBackBone.forward applies torch.relu after each TAGConv call.
        self._layer_outputs.append(torch.relu(output).detach())

    def build_graph(self, env: CoverageSystem, params: cc.Parameters) -> Data:
        """PyG graph exactly as the library builds it for the controller.

        Node features are the four CNN input channels, ``pos`` the normalized
        positions, ``edge_index`` the symmetric radius graph at the current
        communication range.  ``edge_weight`` is present in the object but the
        LPAC forward pass does not pass it to the GNN, so the adjacency is binary.
        """
        return CoverageEnvUtils.get_torch_geometric_data(
            env, params, USE_CNN, USE_COMM_MAP, CNN_MAP_SIZE
        ).to(self.device)

    @torch.no_grad()
    def act(self, data: Data) -> tuple[PointVector, bool, list[torch.Tensor]]:
        """Velocity actions for the current graph, plus convergence flag and layer outputs."""
        self._layer_outputs = []
        actions = self.model(data) * self.actions_std + self.actions_mean
        converged = bool(torch.allclose(actions, torch.zeros_like(actions), atol=1e-5))
        layer_outputs = list(self._layer_outputs)
        return PointVector(actions.cpu().numpy()), converged, layer_outputs

In [ ]:
%%writefile lpac_study/episode.py
"""Run one controller for one episode and record costs and graph statistics."""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np
import coverage_control as cc
from coverage_control import CoverageSystem
from coverage_control.algorithms import CentralizedCVT, ClairvoyantCVT, DecentralizedCVT

from .effective_rank import effective_rank
from .graph_stats import degree_stats
from .lpac_policy import LPACPolicy

CVT_ALGORITHMS = {
    "DecentralizedCVT": DecentralizedCVT,
    "CentralizedCVT": CentralizedCVT,
    "ClairvoyantCVT": ClairvoyantCVT,
}
LPAC_NAME = "lpac"
CONTROLLERS = (LPAC_NAME, *CVT_ALGORITHMS)


@dataclass
class EpisodeRecord:
    """Everything measured during one episode of one controller."""

    costs: np.ndarray  # cost / initial cost, index 0 is the initial state
    converged_step: int | None = None
    degree_mean: float = math.nan
    degree_std: float = math.nan
    isolated_fraction: float = math.nan
    layer_eranks: np.ndarray | None = None  # mean effective rank per GNN layer
    layer_eranks_std: np.ndarray | None = None
    _degrees: list = field(default_factory=list, repr=False)

    @property
    def cost_mean(self) -> float:
        """Average cost over the whole episode (the paper's 'average cost')."""
        return float(self.costs.mean())

    @property
    def cost_final(self) -> float:
        return float(self.costs[-1])


def _fill_after_convergence(costs: np.ndarray, step: int, value: float) -> None:
    """Freeze the cost curve once a controller stops moving, as the library evaluator does."""
    costs[step:] = value


def run_cvt_episode(
    algorithm_name: str, params: cc.Parameters, env: CoverageSystem, num_steps: int
) -> EpisodeRecord:
    """Run a CVT baseline (clairvoyant, centralized or decentralized)."""
    algorithm = CVT_ALGORITHMS[algorithm_name](params, env)
    initial_cost = env.GetObjectiveValue()
    costs = np.ones(num_steps)
    record = EpisodeRecord(costs=costs)
    for step in range(1, num_steps):
        algorithm.ComputeActions()
        if env.StepActions(algorithm.GetActions()):
            raise RuntimeError("Simulator reported an error while stepping")
        costs[step] = env.GetObjectiveValue() / initial_cost
        if algorithm.IsConverged():
            _fill_after_convergence(costs, step, costs[step])
            record.converged_step = step
            break
    return record


def run_lpac_episode(
    policy: LPACPolicy,
    params: cc.Parameters,
    env: CoverageSystem,
    num_steps: int,
    record_rank: bool = True,
) -> EpisodeRecord:
    """Run the LPAC policy and measure degree and effective rank at every step."""
    initial_cost = env.GetObjectiveValue()
    num_robots = env.GetNumRobots()
    costs = np.ones(num_steps)
    record = EpisodeRecord(costs=costs)
    degree_sum = degree_sq_sum = isolated_sum = 0.0
    eranks: list[list[float]] = []
    steps_run = 0
    for step in range(1, num_steps):
        data = policy.build_graph(env, params)
        stats = degree_stats(data.edge_index, num_robots)
        degree_sum += stats.mean
        degree_sq_sum += stats.std**2 + stats.mean**2
        isolated_sum += stats.isolated_fraction
        actions, converged, layer_outputs = policy.act(data)
        if record_rank:
            eranks.append([effective_rank(h) for h in layer_outputs])
        env.StepActions(actions)
        costs[step] = env.GetObjectiveValue() / initial_cost
        steps_run += 1
        if converged:
            _fill_after_convergence(costs, step, costs[step])
            record.converged_step = step
            break
    record.degree_mean = degree_sum / steps_run
    record.degree_std = math.sqrt(max(degree_sq_sum / steps_run - record.degree_mean**2, 0.0))
    record.isolated_fraction = isolated_sum / steps_run
    if record_rank and eranks:
        eranks_array = np.asarray(eranks)
        record.layer_eranks = eranks_array.mean(axis=0)
        record.layer_eranks_std = eranks_array.std(axis=0)
    return record

In [ ]:
%%writefile lpac_study/sweep.py
"""Communication-range sweep of the published LPAC model and the CVT baselines.

Example (full study, as reported):

    python -m lpac_study.sweep --data-dir /path/to/lpac --out-dir results

The script is resumable: rows already present in ``results.csv`` are skipped.
Outputs (all in ``--out-dir``):

* ``results.csv``   one row per (environment, range, robots, controller);
* ``rank.csv``      effective rank per GNN layer for the LPAC rows;
* ``curves.csv``    sub-sampled cost curves, for time-series figures;
* ``run_info.json`` versions, device and timing of the run.
"""
from __future__ import annotations

import argparse
import csv
import json
import platform
import sys
import time
from dataclasses import dataclass
from pathlib import Path

import torch
import torch_geometric
import coverage_control as cc

from .episode import CONTROLLERS, LPAC_NAME, EpisodeRecord, run_cvt_episode, run_lpac_episode
from .lpac_policy import LPACPolicy
from .paths import DataPaths, resolve_data_dir
from .simulation import load_initial_positions, make_environment, make_parameters

DEFAULT_RANGES = (128, 256, 512, 768, 1024)
# (communication range, number of robots) pairs keeping the expected degree
# roughly constant: the number of neighbours within range scales like N * r^2.
DEFAULT_CONSTANT_DEGREE = ("256:32", "512:11", "768:7", "1024:6")
# These two baselines never read the communication range (checked in the C++ source).
RANGE_INDEPENDENT = ("CentralizedCVT", "ClairvoyantCVT")

RESULT_FIELDS = [
    "env", "comm_range", "num_robots", "controller", "sweep",
    "cost_mean", "cost_final", "ratio_mean", "ratio_final",
    "degree_mean", "degree_std", "isolated_fraction", "converged_step",
    "wall_s", "computed",
]
RANK_FIELDS = ["env", "comm_range", "num_robots", "layer", "erank_mean", "erank_std"]
CURVE_FIELDS = ["env", "comm_range", "num_robots", "controller", "step", "cost"]


@dataclass(frozen=True)
class Configuration:
    """One point of a sweep."""

    comm_range: float
    num_robots: int
    sweep: str  # "range" or "constant_degree"


def parse_pair(value: str) -> tuple[int, int]:
    """Parse one ``range:robots`` command-line pair."""
    comm_range, num_robots = value.split(":")
    return int(comm_range), int(num_robots)


def build_configurations(args: argparse.Namespace) -> list[Configuration]:
    """Range sweep at fixed team size, then the constant-degree sweep."""
    configs = [Configuration(float(r), args.num_robots, "range") for r in args.ranges]
    if not args.no_constant_degree:
        pairs = [parse_pair(p) for p in args.constant_degree]
        configs += [Configuration(float(r), n, "constant_degree") for r, n in pairs]
    return configs


def select_device(name: str) -> torch.device:
    """``auto`` picks CUDA when available, else CPU."""
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    return torch.device(name)


@dataclass(frozen=True)
class CostSummary:
    """The two numbers of a clairvoyant run needed to normalize the other controllers."""

    cost_mean: float
    cost_final: float


class ResultWriter:
    """Append-only CSV writers that make the sweep resumable."""

    def __init__(self, out_dir: Path):
        out_dir.mkdir(parents=True, exist_ok=True)
        self.results = out_dir / "results.csv"
        self.rank = out_dir / "rank.csv"
        self.curves = out_dir / "curves.csv"
        self.done = self._read_done_keys()

    def _read_done_keys(self) -> set[tuple]:
        """Keys of rows already in results.csv; clairvoyant costs are kept for normalization."""
        self.clairvoyant: dict[tuple, CostSummary] = {}
        if not self.results.is_file():
            return set()
        done = set()
        with self.results.open(newline="") as handle:
            for r in csv.DictReader(handle):
                key = (int(r["env"]), float(r["comm_range"]), int(r["num_robots"]), r["controller"])
                done.add(key)
                if r["controller"] == "ClairvoyantCVT":
                    self.clairvoyant[(key[0], key[2])] = CostSummary(float(r["cost_mean"]), float(r["cost_final"]))
        return done

    @staticmethod
    def _append(path: Path, fields: list[str], rows: list[dict]) -> None:
        new_file = not path.is_file()
        with path.open("a", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields)
            if new_file:
                writer.writeheader()
            writer.writerows(rows)

    def write_result(self, row: dict) -> None:
        self._append(self.results, RESULT_FIELDS, [row])
        self.done.add((row["env"], row["comm_range"], row["num_robots"], row["controller"]))
        if row["controller"] == "ClairvoyantCVT":
            self.clairvoyant[(row["env"], row["num_robots"])] = CostSummary(row["cost_mean"], row["cost_final"])

    def write_rank(self, rows: list[dict]) -> None:
        if rows:
            self._append(self.rank, RANK_FIELDS, rows)

    def write_curve(self, rows: list[dict]) -> None:
        self._append(self.curves, CURVE_FIELDS, rows)


def evaluate_controller(
    controller: str,
    config: Configuration,
    env_index: int,
    paths: DataPaths,
    policy: LPACPolicy | None,
    num_steps: int,
    record_rank: bool,
) -> EpisodeRecord:
    """Build a fresh environment and run one controller on it."""
    params = make_parameters(paths.env_params, config.comm_range, config.num_robots)
    positions = load_initial_positions(paths.pos_file(env_index), config.num_robots)
    env = make_environment(params, paths.env_file(env_index), positions)
    if controller == LPAC_NAME:
        return run_lpac_episode(policy, params, env, num_steps, record_rank)
    return run_cvt_episode(controller, params, env, num_steps)


def result_row(controller, config, env_index, record, clairvoyant, wall_s, computed) -> dict:
    """One line of ``results.csv``; ratios are relative to the clairvoyant run."""
    return {
        "env": env_index, "comm_range": config.comm_range, "num_robots": config.num_robots,
        "controller": controller, "sweep": config.sweep,
        "cost_mean": record.cost_mean, "cost_final": record.cost_final,
        "ratio_mean": record.cost_mean / clairvoyant.cost_mean,
        "ratio_final": record.cost_final / clairvoyant.cost_final,
        "degree_mean": record.degree_mean, "degree_std": record.degree_std,
        "isolated_fraction": record.isolated_fraction,
        "converged_step": "" if record.converged_step is None else record.converged_step,
        "wall_s": round(wall_s, 2), "computed": computed,
    }


def rank_rows(config, env_index, record) -> list[dict]:
    """Effective rank per GNN layer, averaged over the steps of the episode."""
    if record.layer_eranks is None:
        return []
    return [
        {"env": env_index, "comm_range": config.comm_range, "num_robots": config.num_robots,
         "layer": layer, "erank_mean": float(mean), "erank_std": float(std)}
        for layer, (mean, std) in enumerate(zip(record.layer_eranks, record.layer_eranks_std))
    ]


def curve_rows(controller, config, env_index, record, stride) -> list[dict]:
    """Cost curve sub-sampled every ``stride`` steps (the last step is always kept)."""
    steps = list(range(0, len(record.costs), stride))
    if steps[-1] != len(record.costs) - 1:
        steps.append(len(record.costs) - 1)
    return [
        {"env": env_index, "comm_range": config.comm_range, "num_robots": config.num_robots,
         "controller": controller, "step": s, "cost": float(record.costs[s])}
        for s in steps
    ]


def run_sweep(args: argparse.Namespace) -> None:
    """Main loop: configurations x environments x controllers, written as they finish."""
    paths = resolve_data_dir(args.data_dir)
    device = select_device(args.device)
    torch.manual_seed(args.seed)
    out_dir = Path(args.out_dir)
    writer = ResultWriter(out_dir)
    policy = LPACPolicy(paths, device) if LPAC_NAME in args.controllers else None
    num_envs = min(args.num_envs, paths.count_environments())
    env_indices = range(args.env_start, num_envs if args.env_end is None else min(args.env_end, num_envs))
    cache: dict[tuple, EpisodeRecord] = {}  # (env, robots, controller) of range-independent CVTs
    started = time.time()
    print(f"device={device} envs={num_envs} steps={args.num_steps} out={out_dir}", flush=True)

    # Clairvoyant first: every other controller is normalized by it.
    order = sorted(args.controllers, key=lambda c: c != "ClairvoyantCVT")
    for config in build_configurations(args):
        for env_index in env_indices:
            for controller in order:
                key = (env_index, config.comm_range, config.num_robots, controller)
                if key in writer.done:
                    continue
                cache_key = (env_index, config.num_robots, controller)
                reuse = args.reuse_range_independent and controller in RANGE_INDEPENDENT
                tic = time.time()
                if reuse and cache_key in cache:
                    record, computed = cache[cache_key], False
                else:
                    record = evaluate_controller(
                        controller, config, env_index, paths, policy, args.num_steps, not args.no_rank
                    )
                    cache[cache_key] = record
                    computed = True
                wall = time.time() - tic
                clairvoyant = cache.get((env_index, config.num_robots, "ClairvoyantCVT"))                     or writer.clairvoyant.get((env_index, config.num_robots))
                if clairvoyant is None:
                    raise RuntimeError("ClairvoyantCVT must be evaluated first to normalize costs")
                writer.write_result(result_row(controller, config, env_index, record, clairvoyant, wall, computed))
                writer.write_curve(curve_rows(controller, config, env_index, record, args.curve_stride))
                if controller == LPAC_NAME:
                    writer.write_rank(rank_rows(config, env_index, record))
                print(
                    f"[{time.time() - started:7.0f}s] r={config.comm_range:6.0f} N={config.num_robots:2d} "
                    f"env={env_index:2d} {controller:16s} cost_mean={record.cost_mean:.4f} "
                    f"ratio={record.cost_mean / clairvoyant.cost_mean:.3f} deg={record.degree_mean:5.2f} "
                    f"({wall:5.1f}s{'' if computed else ', reused'})",
                    flush=True,
                )
    write_run_info(out_dir, args, device, time.time() - started)


def write_run_info(out_dir: Path, args: argparse.Namespace, device: torch.device, elapsed: float) -> None:
    """Record software versions and timing next to the results."""
    info = {
        "args": vars(args), "device": str(device), "elapsed_s": round(elapsed, 1),
        "python": sys.version.split()[0], "platform": platform.platform(),
        "torch": torch.__version__, "torch_geometric": torch_geometric.__version__,
        "coverage_control": getattr(cc, "__version__", "unknown"),
    }
    (out_dir / "run_info.json").write_text(json.dumps(info, indent=2))


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter
    )
    parser.add_argument("--data-dir", help="extracted 'lpac' folder (or set $LPAC_DATA_DIR)")
    parser.add_argument("--out-dir", default="results", help="where CSV outputs are written")
    parser.add_argument("--ranges", type=float, nargs="*", default=DEFAULT_RANGES,
                        help="communication ranges in metres for the range sweep (empty list skips it)")
    parser.add_argument("--num-robots", type=int, default=32, help="team size of the range sweep")
    parser.add_argument("--constant-degree", nargs="+", default=DEFAULT_CONSTANT_DEGREE,
                        metavar="RANGE:ROBOTS", help="(range, robots) pairs of the constant-degree sweep")
    parser.add_argument("--no-constant-degree", action="store_true", help="skip the constant-degree sweep")
    parser.add_argument("--num-envs", type=int, default=11, help="number of release environments to use")
    parser.add_argument("--num-steps", type=int, default=600, help="episode length (NumSteps of eval.toml)")
    parser.add_argument("--env-start", type=int, default=0, help="first environment index (to split work)")
    parser.add_argument("--env-end", type=int, default=None, help="one past the last environment index")
    parser.add_argument("--controllers", nargs="+", default=list(CONTROLLERS), choices=CONTROLLERS)
    parser.add_argument("--reuse-range-independent", action="store_true",
                        help="run Centralized/Clairvoyant CVT once per (env, robots) instead of once per range")
    parser.add_argument("--no-rank", action="store_true", help="skip the effective-rank measurement")
    parser.add_argument("--curve-stride", type=int, default=10, help="keep every k-th step of the cost curves")
    parser.add_argument("--device", default="auto", help="auto, cpu or cuda")
    parser.add_argument("--seed", type=int, default=0, help="torch seed (the pipeline is deterministic anyway)")
    return parser


def main(argv: list[str] | None = None) -> None:
    run_sweep(build_parser().parse_args(argv))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lpac_study/merge.py
"""Merge the CSV outputs of several sweep (or factorial) processes and print a summary table.

    python -m lpac_study.merge --parts results/part_a results/part_b --out-dir results
    python -m lpac_study.merge --kind factorial --parts results_2x2/f1024a results_2x2/r768_env_0 --out-dir results_2x2

Rows are de-duplicated on (env, comm_range, num_robots, controller); the
first occurrence wins.  The summary printed at the end is the mean and the
standard deviation across environments of the clairvoyant-normalized cost.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd

KEY = ["env", "comm_range", "num_robots", "controller"]
FILES = {"results.csv": KEY, "rank.csv": ["env", "comm_range", "num_robots", "layer"],
         "curves.csv": KEY + ["step"]}
FACTORIAL_KEY = ["env", "comm_range", "cell", "k", "repeat"]
FACTORIAL_FILES = {"results_2x2.csv": FACTORIAL_KEY}


def merge_file(parts: list[Path], name: str, key: list[str]) -> pd.DataFrame:
    """Concatenate one CSV across parts, dropping duplicate keys."""
    frames = [pd.read_csv(p / name) for p in parts if (p / name).is_file()]
    if not frames:
        return pd.DataFrame()
    merged = pd.concat(frames, ignore_index=True)
    if "repeat" in key:  # files written before the sensitivity repeats existed have no such column
        merged["repeat"] = merged.get("repeat", 0).fillna(0).astype(int)
        merged["noise_sigma"] = merged.get("noise_sigma", 0.0).fillna(0.0)
    return merged.drop_duplicates(subset=key).sort_values(key)


def factorial_summary(results: pd.DataFrame, metric: str = "ratio_mean") -> pd.DataFrame:
    """Mean ± std across environments per (range, cell, k) of the factorial experiment."""
    nominal = results[results["repeat"] == 0] if "repeat" in results else results
    grouped = nominal.groupby(["comm_range", "cell", "k"])
    table = grouped[metric].agg(mean="mean", std="std", n_env="count").reset_index()
    extras = grouped[["degree_mean", "map_neighbours_mean"]].mean().reset_index()
    return table.merge(extras).round(3)


def summary(results: pd.DataFrame, metric: str = "ratio_mean") -> pd.DataFrame:
    """Mean ± std across environments, one row per (sweep, range, robots, controller)."""
    grouped = results.groupby(["sweep", "comm_range", "num_robots", "controller"])[metric]
    table = grouped.agg(mean="mean", std="std", n_env="count").reset_index()
    degrees = results[results["controller"] == "lpac"].groupby(
        ["sweep", "comm_range", "num_robots"])["degree_mean"].mean().rename("degree_lpac")
    return table.merge(degrees.reset_index(), how="left").round(4)


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--parts", nargs="+", required=True, help="output folders of sweep processes")
    parser.add_argument("--out-dir", required=True, help="folder receiving the merged CSVs")
    parser.add_argument("--kind", choices=["sweep", "factorial"], default="sweep",
                        help="sweep: results/rank/curves of lpac_study.sweep; factorial: results_2x2.csv")
    args = parser.parse_args(argv)
    parts = [Path(p) for p in args.parts]
    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    files = FILES if args.kind == "sweep" else FACTORIAL_FILES
    for name, key in files.items():
        merged = merge_file(parts, name, key)
        if not merged.empty:
            merged.to_csv(out_dir / name, index=False)
            print(f"{name}: {len(merged)} rows")
    with pd.option_context("display.width", 200, "display.max_rows", 500):
        if args.kind == "sweep":
            print(summary(pd.read_csv(out_dir / "results.csv")).to_string(index=False))
        else:
            print(factorial_summary(pd.read_csv(out_dir / "results_2x2.csv")).to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lpac_study/plots.py
"""Figures of the study, built only from measured rows of ``results.csv`` / ``rank.csv``.

    python -m lpac_study.plots --results-dir results --fig-dir .

Produces ``fig_portee.png`` (cost vs range, 4 controllers), ``fig_degre.png``
(measured degree vs range), ``fig_degre_constant.png`` (constant-degree sweep)
and ``fig_rang.png`` (effective rank per GNN layer) when the data exist.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from .sweep import DEFAULT_CONSTANT_DEGREE

# Categorical palette, fixed order (validated colour-blind-safe slots).
COLORS = {
    "lpac": "#2a78d6",
    "DecentralizedCVT": "#eb6834",
    "CentralizedCVT": "#1baf7a",
    "ClairvoyantCVT": "#eda100",
}
LABELS = {
    "lpac": "LPAC-K3 (modèle publié, entraîné à 128 m)",
    "DecentralizedCVT": "CVT décentralisé",
    "CentralizedCVT": "CVT centralisé",
    "ClairvoyantCVT": "CVT clairvoyant (référence = 1)",
}
LABELS_EN = {
    "lpac": "LPAC-K3 (published model, trained at 128 m)",
    "DecentralizedCVT": "Decentralized CVT",
    "CentralizedCVT": "Centralized CVT",
    "ClairvoyantCVT": "Clairvoyant CVT (reference = 1)",
}
# Sequential blue ramp (light to dark) for ordered communication ranges.
RANGE_RAMP = ["#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95"]
TEXT = "#333333"
GRID = "#dddddd"
ERRORBAR_STYLE = dict(marker="o", markersize=6, linewidth=2, capsize=4)


def style_axis(ax, xlabel: str, ylabel: str, title: str) -> None:
    """Recessive grid and spines, explicit axis labels with units."""
    ax.set_xlabel(xlabel, color=TEXT)
    ax.set_ylabel(ylabel, color=TEXT)
    ax.set_title(title, fontsize=10, color=TEXT)
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=TEXT)


def aggregate(frame: pd.DataFrame, value: str) -> pd.DataFrame:
    """Mean and standard deviation across environments for one metric."""
    grouped = frame.groupby(["comm_range", "num_robots", "controller"])[value]
    out = grouped.agg(["mean", "std", "count"]).reset_index()
    out["std"] = out["std"].fillna(0.0)
    return out


def count_environments(frame: pd.DataFrame) -> int:
    """Smallest number of environments behind any plotted point."""
    return int(frame.groupby(["comm_range", "num_robots", "controller"])["env"].nunique().min())


def draw_controllers(ax, stats: pd.DataFrame, labels: dict = LABELS) -> None:
    """One error-bar curve per controller, in the fixed palette order."""
    for controller, color in COLORS.items():
        rows = stats[stats["controller"] == controller].sort_values("comm_range")
        if not rows.empty:
            ax.errorbar(rows["comm_range"], rows["mean"], yerr=rows["std"], color=color,
                        label=labels[controller], **ERRORBAR_STYLE)


def plot_cost_vs_range(results: pd.DataFrame, path: Path, metric: str = "ratio_mean", lang: str = "fr") -> None:
    """Normalized cost versus communication range; error bars = std across environments."""
    sweep = results[results["sweep"] == "range"]
    if sweep.empty:
        return
    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
    n_robots = int(sweep["num_robots"].iloc[0])
    n_env = count_environments(sweep)
    if lang == "en":
        draw_controllers(ax, aggregate(sweep, metric), LABELS_EN)
        style_axis(ax, "Communication range at test time (m)",
                   "Coverage cost / clairvoyant cost\n(episode average)",
                   f"Normalized cost versus communication range — {n_robots} robots, {n_env} environments\n"
                   "bars = std across environments; the model was trained at 128 m only")
    else:
        draw_controllers(ax, aggregate(sweep, metric))
        style_axis(
            ax, "Portée de communication (m)",
            "Coût de couverture / coût du clairvoyant\n(moyenne sur l'épisode)",
            f"Coût normalisé selon la portée — {n_robots} robots, {n_env} environnements\n"
            "barres = écart-type entre environnements",
        )
    ax.set_xticks(sorted(sweep["comm_range"].unique()))
    ax.legend(frameon=False, fontsize=9, loc="best")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_degree_vs_range(results: pd.DataFrame, path: Path) -> None:
    """Mean degree of the LPAC communication graph along the trajectories."""
    rows = results[(results["sweep"] == "range") & (results["controller"] == "lpac")]
    if rows.empty:
        return
    stats = aggregate(rows, "degree_mean").sort_values("comm_range")
    n_robots = int(rows["num_robots"].iloc[0])
    fig, ax = plt.subplots(figsize=(7, 4.5), dpi=150)
    ax.errorbar(stats["comm_range"], stats["mean"], yerr=stats["std"], color=COLORS["lpac"],
                label="degré moyen mesuré sur l'edge_index fourni au GNN", **ERRORBAR_STYLE)
    ax.axhline(n_robots - 1, color=GRID, linestyle="--", linewidth=1.5)
    ax.annotate(f"graphe complet ({n_robots - 1} voisins)", (stats["comm_range"].min(), n_robots - 1),
                textcoords="offset points", xytext=(0, 4), fontsize=8, color=TEXT, ha="left")
    for x, y in zip(stats["comm_range"], stats["mean"]):
        ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(-6, 8), fontsize=8,
                    color=TEXT, ha="right")
    style_axis(
        ax, "Portée de communication (m)", "Degré moyen par robot (voisins)",
        f"Degré réel du graphe de communication le long des trajectoires LPAC ({n_robots} robots)\n"
        "moyenne sur robots, pas et environnements ; barres = écart-type entre environnements",
    )
    ax.set_xticks(sorted(rows["comm_range"].unique()))
    ax.set_ylim(0, n_robots + 1)
    ax.legend(frameon=False, fontsize=9, loc="center left")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def select_constant_degree(results: pd.DataFrame) -> pd.DataFrame:
    """Rows of the constant-degree sweep.

    The (range, robots) pairs are those labelled ``constant_degree`` plus any
    pair of the default list that the range sweep already covers (for example
    (256 m, 32 robots) is shared by both sweeps and is stored once).
    """
    labelled = results[results["sweep"] == "constant_degree"]
    pairs = {(float(r), int(n)) for r, n in zip(labelled["comm_range"], labelled["num_robots"])}
    pairs |= {(float(p.split(":")[0]), int(p.split(":")[1])) for p in DEFAULT_CONSTANT_DEGREE}
    mask = [(float(r), int(n)) in pairs for r, n in zip(results["comm_range"], results["num_robots"])]
    return results[mask]


def plot_constant_degree(results: pd.DataFrame, path: Path, metric: str = "ratio_mean") -> None:
    """Range sweep at (approximately) constant degree, team size varying with range."""
    sweep = select_constant_degree(results)
    if sweep.empty:
        return
    degrees = aggregate(sweep[sweep["controller"] == "lpac"], "degree_mean").sort_values("comm_range")
    fig, (ax, ax_deg) = plt.subplots(2, 1, figsize=(8, 7), dpi=150, sharex=True,
                                     gridspec_kw={"height_ratios": [3, 1.3]})
    draw_controllers(ax, aggregate(sweep, metric))
    ax_deg.errorbar(degrees["comm_range"], degrees["mean"], yerr=degrees["std"], color=COLORS["lpac"],
                    **ERRORBAR_STYLE)
    for _, row in degrees.iterrows():
        ax_deg.annotate(f"{row['mean']:.1f}", (row["comm_range"], row["mean"]),
                        textcoords="offset points", xytext=(8, 6), fontsize=8, color=TEXT)
    pairs = sweep[["comm_range", "num_robots"]].drop_duplicates().sort_values("comm_range")
    ax_deg.set_xticks(list(pairs["comm_range"]))
    ax_deg.set_xticklabels([f"{int(r)} m\n{int(n)} robots" for r, n in zip(pairs["comm_range"], pairs["num_robots"])])
    style_axis(
        ax, "", "Coût / coût du clairvoyant\n(moyenne sur l'épisode)",
        f"Balayage à degré approximativement constant — {count_environments(sweep)} environnements\n"
        "barres = écart-type entre environnements",
    )
    style_axis(ax_deg, "Portée de communication et taille de l'équipe", "Degré moyen\n(LPAC)", "")
    ax.legend(frameon=False, fontsize=9, loc="best")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_effective_rank(rank: pd.DataFrame, results: pd.DataFrame, path: Path) -> None:
    """Effective rank of each TAGConv layer output, one curve per communication range."""
    main = results[(results["sweep"] == "range") & (results["controller"] == "lpac")]
    if rank.empty or main.empty:
        return
    n_robots = int(main["num_robots"].iloc[0])
    rank = rank[rank["num_robots"] == n_robots]
    stats = rank.groupby(["comm_range", "layer"])["erank_mean"].agg(["mean", "std"]).reset_index()
    ranges = sorted(stats["comm_range"].unique())
    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
    for color, comm_range in zip(RANGE_RAMP, ranges):
        rows = stats[stats["comm_range"] == comm_range].sort_values("layer")
        ax.errorbar(rows["layer"] + 1, rows["mean"], yerr=rows["std"].fillna(0.0), color=color,
                    label=f"{int(comm_range)} m", **ERRORBAR_STYLE)
    n_env = int(rank.groupby("comm_range")["env"].nunique().min())
    style_axis(
        ax, "Couche TAGConv (sortie, après ReLU)",
        "Rang effectif des représentations des robots\n(exp. de l'entropie des valeurs singulières)",
        f"Rang effectif par couche selon la portée — LPAC, {n_robots} robots, {n_env} environnements\n"
        f"moyenne sur les pas de temps ; barres = écart-type entre environnements ; maximum = {n_robots}",
    )
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_ylim(bottom=0)
    ax.legend(title="Portée", frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def load_table(path: Path) -> pd.DataFrame:
    return pd.read_csv(path) if path.is_file() else pd.DataFrame()


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--results-dir", default="results", help="folder holding results.csv and rank.csv")
    parser.add_argument("--fig-dir", default=".", help="where PNG figures are written")
    args = parser.parse_args(argv)
    results_dir, fig_dir = Path(args.results_dir), Path(args.fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)
    results = load_table(results_dir / "results.csv")
    rank = load_table(results_dir / "rank.csv")
    if results.empty:
        raise SystemExit(f"No results.csv in {results_dir}")
    plot_cost_vs_range(results, fig_dir / "fig_portee.png")
    plot_cost_vs_range(results, fig_dir / "fig_range_en.png", lang="en")
    plot_degree_vs_range(results, fig_dir / "fig_degre.png")
    plot_constant_degree(results, fig_dir / "fig_degre_constant.png")
    plot_effective_rank(rank, results, fig_dir / "fig_rang.png")
    print("figures written to", fig_dir)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lpac_study/neighbourhood.py
"""Neighbourhood construction for the inference-time factorial experiment.

The neighbourhood enters LPAC through two inputs that this module rebuilds
from raw robot positions:

* the communication graph given to the five TAGConv layers (``edge_index``);
* the two "communication map" channels given to the CNN, which encode the
  relative position of neighbours on a ``map_size x map_size`` grid whose
  resolution scales with the communication range.

The map reconstruction mirrors ``CoverageEnvUtils.get_communication_maps`` of
coverage_control 1.3.1 (a robot ``j`` is a neighbour of ``i`` when
``|p_j - p_i| < range``; cell index ``round(rel * map_size / (2 range) +
map_size / 2 - 1/2)``; cell value ``rel / range`` summed over the robots that
fall in the same cell).  ``checks.py`` verifies the reconstruction against the
library before any neighbour restriction is applied.
"""
from __future__ import annotations

import torch
from torch_geometric.utils import dense_to_sparse, to_undirected


def robot_positions_double(env) -> torch.Tensor:
    """Raw robot positions in float64, exactly as the C++ simulator holds them.

    The library converts positions to float32 before use; computing relative
    positions in double and casting afterwards reproduces the C++ maps bit for bit.
    """
    return torch.tensor([[p[0], p[1]] for p in env.GetRobotPositions()], dtype=torch.float64)


def relative_positions(positions: torch.Tensor) -> torch.Tensor:
    """``rel[i, j] = p_j - p_i`` for raw positions of shape ``(N, 2)``."""
    return positions.unsqueeze(0) - positions.unsqueeze(1)


def in_range_mask(positions: torch.Tensor, comm_range: float) -> torch.Tensor:
    """Boolean ``(N, N)`` mask of robots strictly within communication range (no self-loops)."""
    distances = torch.cdist(positions, positions)
    mask = distances < comm_range
    mask.fill_diagonal_(False)
    return mask


def knn_within_range_mask(positions: torch.Tensor, comm_range: float, k: int) -> torch.Tensor:
    """Directed mask keeping, for every robot, its ``k`` nearest robots among those in range.

    Robots with fewer than ``k`` neighbours in range keep all of them.  This is
    the k-nearest-neighbour rule of Muthusamy et al. (arXiv:2409.19829)
    restricted to physically reachable robots.
    """
    distances = torch.cdist(positions, positions)
    reachable = in_range_mask(positions, comm_range)
    distances = distances.masked_fill(~reachable, float("inf"))
    k = min(k, positions.shape[0] - 1)
    nearest = distances.topk(k, dim=1, largest=False).indices
    mask = torch.zeros_like(reachable)
    mask.scatter_(1, nearest, True)
    return mask & reachable


def symmetrize(mask: torch.Tensor) -> torch.Tensor:
    """Undirected version of a neighbour mask: keep an edge if either robot selected the other."""
    return mask | mask.T


def mask_to_edge_index(mask: torch.Tensor) -> torch.Tensor:
    """PyG ``edge_index`` (undirected, coalesced) from a symmetric boolean mask."""
    edge_index, _ = dense_to_sparse(mask.to(torch.float32))
    return to_undirected(edge_index, num_nodes=mask.shape[0])


def comm_maps_from_positions(
    positions: torch.Tensor, neighbour_mask: torch.Tensor, comm_range: float, map_size: int
) -> torch.Tensor:
    """Communication maps ``(N, 2, map_size, map_size)`` for the neighbours of ``neighbour_mask``.

    Args:
        positions: raw robot positions, shape ``(N, 2)``, in metres.
        neighbour_mask: ``(N, N)`` boolean, row ``i`` = robots drawn on robot ``i``'s map.
        comm_range: range used for the grid resolution and the value normalization.
        map_size: side of the square map (32 for the published model).
    """
    num_robots = positions.shape[0]
    rel = relative_positions(positions).to(torch.float32)  # the library works on float32 relative positions
    scaled = torch.round(rel * map_size / (comm_range * 2.0) + (map_size / 2.0 - 0.5)).long()
    values = rel / comm_range
    maps = torch.zeros((num_robots, 2, map_size, map_size), dtype=torch.float32)
    robot, neighbour = neighbour_mask.nonzero(as_tuple=True)
    cell_x, cell_y = scaled[robot, neighbour, 0], scaled[robot, neighbour, 1]
    if len(robot) and (cell_x.min() < 0 or cell_y.min() < 0 or cell_x.max() >= map_size or cell_y.max() >= map_size):
        raise ValueError("a neighbour falls outside the communication map")
    for channel in (0, 1):
        maps[:, channel].index_put_((robot, cell_x, cell_y), values[robot, neighbour, channel], accumulate=True)
    return maps

In [ ]:
%%writefile lpac_study/checks.py
"""Mandatory checks before the factorial experiment: the rebuilt inputs match the library.

    python -m lpac_study.checks --data-dir /path/to/lpac [--comm-range 1024] [--steps 20]

Runs the published policy for a few steps at the given range and, at every
step, compares against ``CoverageEnvUtils.get_torch_geometric_data``:

1. the communication-map channels rebuilt from positions with all in-range
   neighbours (maximum absolute error reported);
2. which channels of ``data.x`` are communication maps (1 and 2) and which are
   perception maps (0: local importance map, 3: obstacle map), by checking
   that only channels 1 and 2 change when neighbours are removed;
3. the library's ``edge_index`` equals the in-range graph, and the simulator's
   own neighbour count matches it (the simulated range is untouched).
"""
from __future__ import annotations

import argparse

import torch
from coverage_control import CoverageEnvUtils
from torch_geometric.utils import to_undirected

from .lpac_policy import CNN_MAP_SIZE, LPACPolicy
from .neighbourhood import (comm_maps_from_positions, in_range_mask, knn_within_range_mask, mask_to_edge_index,
                            robot_positions_double, symmetrize)
from .paths import resolve_data_dir
from .simulation import load_initial_positions, make_environment, make_parameters

COMM_CHANNELS = (1, 2)


def edge_set(edge_index: torch.Tensor) -> set[tuple[int, int]]:
    return set(map(tuple, edge_index.t().tolist()))


def run_checks(args: argparse.Namespace) -> None:
    paths = resolve_data_dir(args.data_dir)
    params = make_parameters(paths.env_params, args.comm_range, args.num_robots)
    env = make_environment(params, paths.env_file(args.env), load_initial_positions(paths.pos_file(args.env), args.num_robots))
    policy = LPACPolicy(paths, torch.device("cpu"))
    worst_map_error = 0.0
    worst_other_change = 0.0
    max_lonely = 0
    for step in range(args.steps):
        data = policy.build_graph(env, params)
        positions = robot_positions_double(env)
        full_mask = in_range_mask(positions, params.pCommunicationRange)
        rebuilt = comm_maps_from_positions(positions, full_mask, params.pCommunicationRange, CNN_MAP_SIZE)
        worst_map_error = max(worst_map_error, float((data.x[:, list(COMM_CHANNELS)] - rebuilt).abs().max()))
        # Channels that must not depend on neighbours: compare with a restricted neighbour set.
        restricted = comm_maps_from_positions(positions, symmetrize(knn_within_range_mask(positions, params.pCommunicationRange, 2)),
                                              params.pCommunicationRange, CNN_MAP_SIZE)
        assert not torch.equal(restricted, rebuilt) or full_mask.sum() <= 2 * len(positions)
        other = [c for c in range(data.x.shape[1]) if c not in COMM_CHANNELS]
        worst_other_change = max(worst_other_change, float(data.x[:, other].abs().sum() == 0))
        # Graph: library edge_index (d <= range) versus strict in-range mask.
        library_edges = edge_set(to_undirected(data.edge_index))
        rebuilt_edges = edge_set(mask_to_edge_index(full_mask))
        if library_edges != rebuilt_edges:
            raise AssertionError(f"step {step}: edge sets differ ({len(library_edges ^ rebuilt_edges)} edges)")
        simulator_counts = [len(env.GetRelativePositonsNeighbors(i)) for i in range(len(positions))]
        if simulator_counts != full_mask.sum(1).tolist():
            raise AssertionError(f"step {step}: simulator neighbour counts differ from the in-range mask")
        # A robot without neighbour must have all-zero communication channels in the library's map.
        lonely = [i for i, n in enumerate(simulator_counts) if n == 0]
        if lonely and float(data.x[lonely][:, list(COMM_CHANNELS)].abs().max()) != 0.0:
            raise AssertionError(f"step {step}: a robot without neighbour has a non-zero communication map")
        max_lonely = max(max_lonely, len(lonely))
        actions, _, _ = policy.act(data)
        env.StepActions(actions)
    print(f"steps checked: {args.steps} at {args.comm_range:.0f} m, {args.num_robots} robots, env {args.env}")
    print(f"comm-map reconstruction, max |error| over all steps: {worst_map_error:.3e}")
    print(f"x channels: {data.x.shape[1]} (comm maps = {COMM_CHANNELS}; others = perception maps, never zero: "
          f"{'ok' if worst_other_change == 0 else 'EMPTY'})")
    print(f"edge_index equals the in-range graph and the simulator's neighbour lists: ok")
    print(f"robots without neighbour have all-zero map channels: ok "
          f"(up to {max_lonely} such robots in one step)")
    print(f"simulated communication range after the checks: {params.pCommunicationRange:.0f} m")


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir")
    parser.add_argument("--comm-range", type=float, default=1024.0)
    parser.add_argument("--num-robots", type=int, default=32)
    parser.add_argument("--env", type=int, default=0)
    parser.add_argument("--steps", type=int, default=20)
    run_checks(parser.parse_args(argv))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lpac_study/factorial.py
"""Inference-time factorial experiment: which input carries the neighbourhood that breaks LPAC?

    python -m lpac_study.factorial --data-dir /path/to/lpac --baseline results/results.csv \
        --out-dir results_2x2 --ranges 1024 --cells G0C0 G0C1 G0C2 G1C0 G1C1 G1C2 --k 4

Factor GRAPH (what the GNN sees through ``edge_index``):
  G0  every edge within range (the published behaviour);
  G1  each robot keeps its k nearest robots among those in range, made undirected.
Factor MAP (what the CNN sees through the two communication-map channels):
  C0  the library's map (all robots in range, resolution of the test range);
  C1  only the k nearest neighbours of G1 (same undirected set), test-range resolution;
  C2  only robots within 128 m, drawn with the 128 m normalization and resolution (training scale);
  C3  only robots within 256 m, drawn with the 256 m normalization and resolution;
  C4  empty map: both neighbour channels set to zero (the library's value for "no neighbour").
The simulator itself always runs at the test range; only the model inputs change.
Costs are normalized by the clairvoyant rows of ``--baseline`` (range-independent).

Sensitivity repeats (``--repeats N --noise-sigma 1``): the initial positions are shifted
by Gaussian noise with a fixed seed per (environment, repeat), and the clairvoyant is
re-run on the same shifted start to normalize.  ``repeat = 0`` is the nominal episode.

Outputs ``results_2x2.csv`` (one row per environment, range, cell, k, repeat) in
``--out-dir``; the script is resumable.
"""
from __future__ import annotations

import argparse
import csv
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data

from .episode import run_cvt_episode, run_lpac_episode
from .lpac_policy import CNN_MAP_SIZE, LPACPolicy
from .neighbourhood import (comm_maps_from_positions, in_range_mask, knn_within_range_mask, mask_to_edge_index,
                            robot_positions_double, symmetrize)
from .paths import DataPaths, resolve_data_dir
from .simulation import load_initial_positions, make_environment, make_parameters

NUM_ROBOTS = 32
COMM_CHANNELS = [1, 2]
MAP_SCALE_LEVELS = {"C2": 128.0, "C3": 256.0}  # map drawn as if the range were this value
CELLS = tuple(f"G{g}C{c}" for g in (0, 1) for c in range(5))
FIELDS = ["env", "comm_range", "graph", "map", "cell", "k", "repeat", "noise_sigma",
          "cost_mean", "cost_final", "ratio_mean", "ratio_final", "clairvoyant_cost_mean",
          "degree_mean", "degree_std", "map_neighbours_mean", "converged_step", "wall_s"]
WORLD_MARGIN = 0.5  # metres kept from the world boundary when perturbing initial positions


class FactorialPolicy(LPACPolicy):
    """The published policy whose graph and communication maps are rebuilt per factor level."""

    def __init__(self, paths, device, graph: str, comm_map: str, k: int):
        super().__init__(paths, device)
        self.graph, self.comm_map, self.k = graph, comm_map, k
        self.map_neighbour_counts: list[float] = []

    def build_graph(self, env, params) -> Data:
        data = super().build_graph(env, params)
        positions = robot_positions_double(env)
        test_range = params.pCommunicationRange
        capped = symmetrize(knn_within_range_mask(positions, test_range, self.k))
        if self.graph == "G1":
            data.edge_index = mask_to_edge_index(capped)
        map_mask = self._map_mask(positions, test_range, capped)
        if self.comm_map == "C4":
            data.x[:, COMM_CHANNELS] = 0.0
        elif self.comm_map != "C0":
            map_range = MAP_SCALE_LEVELS.get(self.comm_map, test_range)
            data.x[:, COMM_CHANNELS] = comm_maps_from_positions(positions, map_mask, map_range, CNN_MAP_SIZE)
        self.map_neighbour_counts.append(float(map_mask.sum(1).float().mean()))
        return data

    def _map_mask(self, positions, test_range, capped):
        """Robots drawn on each robot's communication map for the current MAP level."""
        if self.comm_map == "C1":
            return capped
        if self.comm_map in MAP_SCALE_LEVELS:
            return in_range_mask(positions, MAP_SCALE_LEVELS[self.comm_map])
        if self.comm_map == "C4":
            return torch.zeros_like(capped)
        return in_range_mask(positions, test_range)


def load_clairvoyant(baseline: Path, num_robots: int) -> dict[int, tuple[float, float]]:
    """Per-environment clairvoyant (cost_mean, cost_final) from an existing results.csv."""
    rows = pd.read_csv(baseline)
    rows = rows[(rows["controller"] == "ClairvoyantCVT") & (rows["num_robots"] == num_robots)]
    rows = rows.drop_duplicates("env")
    return {int(r.env): (float(r.cost_mean), float(r.cost_final)) for r in rows.itertuples()}


def done_keys(path: Path) -> set[tuple]:
    """Keys (env, range, cell, k, repeat) already present in a results file."""
    if not path.is_file():
        return set()
    with path.open(newline="") as handle:
        return {(int(r["env"]), float(r["comm_range"]), r["cell"], int(r["k"]), int(r.get("repeat") or 0))
                for r in csv.DictReader(handle)}


def append_row(path: Path, row: dict) -> None:
    new = not path.is_file()
    with path.open("a", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=FIELDS)
        if new:
            writer.writeheader()
        writer.writerow(row)


def perturbed_positions(positions: np.ndarray, env_index: int, repeat: int, sigma: float, world_size: float) -> np.ndarray:
    """Initial positions shifted by N(0, sigma) metres with a fixed seed; repeat 0 is unchanged."""
    if repeat == 0 or sigma <= 0:
        return positions
    rng = np.random.default_rng(1000 * env_index + repeat)
    shifted = positions + rng.normal(0.0, sigma, size=positions.shape)
    return np.clip(shifted, WORLD_MARGIN, world_size - WORLD_MARGIN)


def run_cell(paths: DataPaths, env_index: int, comm_range: float, cell: str, k: int, repeat: int,
             sigma: float, num_steps: int, clairvoyant: dict, device) -> dict:
    """One LPAC episode for one factorial cell on one (possibly perturbed) environment."""
    policy = FactorialPolicy(paths, device, cell[:2], cell[2:], k)
    params = make_parameters(paths.env_params, comm_range, NUM_ROBOTS)
    positions = load_initial_positions(paths.pos_file(env_index), NUM_ROBOTS)
    positions = perturbed_positions(positions, env_index, repeat, sigma, params.pWorldMapSize)
    if repeat == 0:
        clair_mean, clair_final = clairvoyant[env_index]
    else:  # the shifted start changes the reference too: re-run the clairvoyant on it
        reference = run_cvt_episode("ClairvoyantCVT", params, make_environment(params, paths.env_file(env_index), positions), num_steps)
        clair_mean, clair_final = reference.cost_mean, reference.cost_final
    env = make_environment(params, paths.env_file(env_index), positions)
    tic = time.time()
    record = run_lpac_episode(policy, params, env, num_steps, record_rank=False)
    return {
        "env": env_index, "comm_range": comm_range, "graph": cell[:2], "map": cell[2:], "cell": cell, "k": k,
        "repeat": repeat, "noise_sigma": sigma if repeat else 0.0,
        "cost_mean": record.cost_mean, "cost_final": record.cost_final,
        "ratio_mean": record.cost_mean / clair_mean, "ratio_final": record.cost_final / clair_final,
        "clairvoyant_cost_mean": clair_mean,
        "degree_mean": record.degree_mean, "degree_std": record.degree_std,
        "map_neighbours_mean": sum(policy.map_neighbour_counts) / len(policy.map_neighbour_counts),
        "converged_step": "" if record.converged_step is None else record.converged_step,
        "wall_s": round(time.time() - tic, 1),
    }


def run_factorial(args: argparse.Namespace) -> None:
    paths = resolve_data_dir(args.data_dir)
    device = torch.device("cpu")
    torch.manual_seed(0)
    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    results = out / "results_2x2.csv"
    done = done_keys(results)
    clairvoyant = load_clairvoyant(Path(args.baseline), NUM_ROBOTS)
    env_end = args.num_envs if args.env_end is None else min(args.env_end, args.num_envs)
    repeats = range(args.repeat_start, args.repeats + 1)
    started = time.time()
    for comm_range in args.ranges:
        for env_index in range(args.env_start, env_end):
            for cell in args.cells:
                for k in args.k:
                    if cell[2:] in ("C0", "C2", "C3", "C4") and cell[:2] == "G0" and k != args.k[0]:
                        continue  # these cells do not depend on k: computed once
                    for repeat in repeats:
                        key = (env_index, float(comm_range), cell, int(k), repeat)
                        if key in done:
                            continue
                        row = run_cell(paths, env_index, float(comm_range), cell, int(k), repeat,
                                       args.noise_sigma, args.num_steps, clairvoyant, device)
                        append_row(results, row)
                        done.add(key)
                        print(f"[{time.time() - started:6.0f}s] r={comm_range:5.0f} env={env_index:2d} {cell} k={k:2d} "
                              f"rep={repeat} ratio={row['ratio_mean']:.3f} deg={row['degree_mean']:5.2f} "
                              f"map_nb={row['map_neighbours_mean']:5.2f} ({row['wall_s']:.0f}s)", flush=True)


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--data-dir")
    parser.add_argument("--baseline", required=True, help="results.csv of the range sweep (clairvoyant normalization)")
    parser.add_argument("--out-dir", default="results_2x2")
    parser.add_argument("--ranges", type=float, nargs="+", default=[1024.0])
    parser.add_argument("--cells", nargs="+", default=list(CELLS[:3]) + list(CELLS[5:8]), choices=CELLS)
    parser.add_argument("--k", type=int, nargs="+", default=[4], help="neighbour caps for G1/C1")
    parser.add_argument("--num-envs", type=int, default=11)
    parser.add_argument("--env-start", type=int, default=0)
    parser.add_argument("--env-end", type=int, default=None)
    parser.add_argument("--num-steps", type=int, default=600)
    parser.add_argument("--repeats", type=int, default=0, help="number of perturbed repeats per episode (0 = nominal only)")
    parser.add_argument("--repeat-start", type=int, default=0, help="first repeat index to run (1 skips the nominal episode)")
    parser.add_argument("--noise-sigma", type=float, default=1.0, help="std (m) of the initial-position noise of repeats")
    return parser


def main(argv: list[str] | None = None) -> None:
    run_factorial(build_parser().parse_args(argv))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile lpac_study/plots_factorial.py
"""Figures of the inference-time factorial experiment (English labels, for the blog post).

    python -m lpac_study.plots_factorial --results results_2x2/results_2x2.csv \
        --baseline results/results.csv --fig-dir .

* ``fig_2x2.png``  the six cells at the main range (mean ± std across environments);
* ``fig_dose.png`` cost versus neighbour cap k for the best cell (dose-response);
* ``fig_fix.png``  the best cell across communication ranges, against the
  published behaviour and the CVT baselines of the range sweep;
* ``fig_scale256.png`` the published model, G1C2, G0C3 and the CVTs across ranges.
Each figure is drawn only when the corresponding rows exist.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from .plots import COLORS, ERRORBAR_STYLE, GRID, TEXT, style_axis

CELL_ORDER = ["G0C0", "G0C1", "G0C2", "G1C0", "G1C1", "G1C2"]
CELL_LABELS = {
    "G0C0": "G0 + C0\nfull graph,\nfull map",
    "G0C1": "G0 + C1\nfull graph,\nmap: k nearest",
    "G0C2": "G0 + C2\nfull graph,\nmap: 128 m",
    "G1C0": "G1 + C0\ngraph: k nearest,\nfull map",
    "G1C1": "G1 + C1\ngraph & map:\nk nearest",
    "G1C2": "G1 + C2\ngraph: k nearest,\nmap: 128 m",
}
CELL_COLORS = {"G0": COLORS["DecentralizedCVT"], "G1": COLORS["lpac"]}
BASELINE_LABELS = {"DecentralizedCVT": "Decentralized CVT", "CentralizedCVT": "Centralized CVT"}
Y_LABEL = "Coverage cost / clairvoyant cost\n(episode average)"


def cell_stats(rows: pd.DataFrame, metric: str = "ratio_mean") -> pd.DataFrame:
    grouped = rows.groupby("cell")[metric].agg(["mean", "std", "count"]).reindex(CELL_ORDER).dropna()
    return grouped


def plot_cells(rows: pd.DataFrame, baseline: pd.DataFrame, path: Path, comm_range: float, k: int) -> None:
    """Dot-and-error-bar chart of the six factorial cells at one range."""
    sel = rows[(rows["comm_range"] == comm_range) & ((rows["k"] == k) | (rows["cell"] == "G0C0"))]
    if sel.empty:
        return
    stats = cell_stats(sel)
    fig, ax = plt.subplots(figsize=(9, 5.2), dpi=150)
    x = np.arange(len(stats))
    for i, (cell, row) in enumerate(stats.iterrows()):
        ax.errorbar(i, row["mean"], yerr=row["std"], color=CELL_COLORS[cell[:2]], **ERRORBAR_STYLE)
        ax.annotate(f"{row['mean']:.2f}", (i, row["mean"]), textcoords="offset points", xytext=(10, 4), fontsize=9, color=TEXT)
    base = baseline[(baseline["comm_range"] == comm_range) & (baseline["num_robots"] == 32)]
    for ctrl, style in (("DecentralizedCVT", "--"), ("CentralizedCVT", ":")):
        value = base[base["controller"] == ctrl]["ratio_mean"].mean()
        if not np.isnan(value):
            ax.axhline(value, color=COLORS[ctrl], linestyle=style, linewidth=1.5, label=BASELINE_LABELS[ctrl])
    ax.axhline(1.0, color=COLORS["ClairvoyantCVT"], linewidth=1.5, label="Clairvoyant CVT (= 1)")
    ax.set_xticks(x)
    ax.set_xticklabels([CELL_LABELS[c] for c in stats.index], fontsize=8.5)
    n_env = int(sel.groupby("cell")["env"].nunique().min())
    style_axis(ax, "", Y_LABEL,
               f"Where does the neighbourhood break LPAC? Inference-time factorial at {comm_range:.0f} m, "
               f"32 robots, k = {k}\n{n_env} environments; bars = std across environments; "
               "blue = capped graph (G1), orange = full graph (G0)")
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, fontsize=9, loc="upper right")
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_dose(rows: pd.DataFrame, path: Path, comm_range: float, cell: str) -> None:
    """Cost versus the neighbour cap k for one cell; the uncapped model is the last point."""
    capped = rows[(rows["comm_range"] == comm_range) & (rows["cell"] == cell)]
    full = rows[(rows["comm_range"] == comm_range) & (rows["cell"] == "G0C0")]
    if capped["k"].nunique() < 2:
        return
    stats = capped.groupby("k")["ratio_mean"].agg(["mean", "std"]).sort_index()
    degrees = capped.groupby("k")["degree_mean"].mean()
    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
    positions = list(range(len(stats)))
    ax.errorbar(positions, stats["mean"], yerr=stats["std"], color=COLORS["lpac"], label=f"{cell}: k nearest neighbours", **ERRORBAR_STYLE)
    labels = [f"k = {int(k)}\n({degrees[k]:.1f} nb.)" for k in stats.index]
    if not full.empty:
        positions.append(len(positions))
        ax.errorbar(positions[-1], full["ratio_mean"].mean(), yerr=full["ratio_mean"].std(), color=COLORS["DecentralizedCVT"],
                    label="G0C0: all neighbours in range", **ERRORBAR_STYLE)
        labels.append(f"all\n({full['degree_mean'].mean():.1f} nb.)")
    ax.set_xticks(positions)
    ax.set_xticklabels(labels, fontsize=9)
    n_env = int(capped.groupby("k")["env"].nunique().min())
    style_axis(ax, "Neighbour cap k (mean degree actually seen by the GNN)", Y_LABEL,
               f"Dose-response of the neighbour cap at {comm_range:.0f} m, 32 robots, {n_env} environments\n"
               "bars = std across environments")
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_fix(rows: pd.DataFrame, baseline: pd.DataFrame, path: Path, cell: str, k: int) -> None:
    """The chosen cell across ranges, next to the published model and the CVT baselines."""
    fixed = rows[(rows["cell"] == cell) & (rows["k"] == k)]
    if fixed["comm_range"].nunique() < 2:
        return
    sweep = baseline[(baseline["sweep"] == "range") & (baseline["num_robots"] == 32)]
    fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
    for ctrl, label in (("lpac", "LPAC-K3 as published (all neighbours)"),
                        ("DecentralizedCVT", "Decentralized CVT"), ("CentralizedCVT", "Centralized CVT"),
                        ("ClairvoyantCVT", "Clairvoyant CVT (= 1)")):
        s = sweep[sweep["controller"] == ctrl].groupby("comm_range")["ratio_mean"].agg(["mean", "std"]).sort_index()
        ax.errorbar(s.index, s["mean"], yerr=s["std"].fillna(0), color=COLORS[ctrl], label=label, alpha=0.55 if ctrl == "lpac" else 1.0,
                    linestyle="--" if ctrl == "lpac" else "-", **ERRORBAR_STYLE)
    f = fixed.groupby("comm_range")["ratio_mean"].agg(["mean", "std"]).sort_index()
    ax.errorbar(f.index, f["mean"], yerr=f["std"].fillna(0), color=COLORS["lpac"], label=f"LPAC-K3 + {cell}, k = {k}", **ERRORBAR_STYLE)
    n_env = int(fixed.groupby("comm_range")["env"].nunique().min())
    style_axis(ax, "Communication range at test time (m)", Y_LABEL,
               f"Inference-time fix across ranges — 32 robots, {n_env} environments, no retraining\n"
               "bars = std across environments")
    ax.set_xticks(sorted(sweep["comm_range"].unique()))
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, fontsize=9)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


SCALE_SERIES = (  # (cell or controller, label, colour, linestyle)
    ("lpac", "LPAC-K3 as published (all neighbours in range)", COLORS["lpac"], "--"),
    ("G1C2", "LPAC-K3 + G1C2: map at the 128 m scale, graph capped (k = 4)", "#184f95", "-"),
    ("G0C3", "LPAC-K3 + G0C3: map at the 256 m scale, full graph", "#3987e5", "-"),
    ("DecentralizedCVT", "Decentralized CVT", COLORS["DecentralizedCVT"], "-"),
    ("CentralizedCVT", "Centralized CVT", COLORS["CentralizedCVT"], "-"),
    ("ClairvoyantCVT", "Clairvoyant CVT (= 1)", COLORS["ClairvoyantCVT"], "-"),
)


def plot_scale(rows: pd.DataFrame, baseline: pd.DataFrame, path: Path, k: int = 4) -> None:
    """Published model, two inference-time map fixes and the CVT baselines across ranges."""
    nominal = rows[(rows.get("repeat", 0) == 0) & ((rows["k"] == k) | rows["cell"].str.startswith("G0"))]
    sweep = baseline[(baseline["sweep"] == "range") & (baseline["num_robots"] == 32)]
    fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=150)
    drawn = 0
    for name, label, color, style in SCALE_SERIES:
        source = sweep[sweep["controller"] == name] if name in COLORS else nominal[nominal["cell"] == name]
        if source.empty:
            continue
        s = source.groupby("comm_range")["ratio_mean"].agg(["mean", "std"]).sort_index()
        ax.errorbar(s.index, s["mean"], yerr=s["std"].fillna(0), color=color, label=label, linestyle=style,
                    alpha=0.6 if name == "lpac" else 1.0, **ERRORBAR_STYLE)
        drawn += 1
    if drawn == 0:
        plt.close(fig)
        return
    n_env = int(nominal.groupby(["comm_range", "cell"])["env"].nunique().min())
    style_axis(ax, "Communication range at test time (m)", Y_LABEL,
               f"Neighbour map drawn at a fixed scale — 32 robots, {n_env} environments, no retraining\n"
               "bars = std across environments")
    ax.set_xticks(sorted(sweep["comm_range"].unique()))
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, fontsize=8.5)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def main(argv: list[str] | None = None) -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--results", required=True, help="results_2x2.csv")
    parser.add_argument("--baseline", required=True, help="results.csv of the range sweep")
    parser.add_argument("--fig-dir", default=".")
    parser.add_argument("--comm-range", type=float, default=1024.0)
    parser.add_argument("--k", type=int, default=4)
    parser.add_argument("--best-cell", default="G1C2", help="cell drawn across ranges in fig_fix.png")
    parser.add_argument("--dose-cell", default="G1C1", help="cell whose k is varied in fig_dose.png")
    args = parser.parse_args(argv)
    rows = pd.read_csv(args.results)
    baseline = pd.read_csv(args.baseline)
    fig_dir = Path(args.fig_dir)
    fig_dir.mkdir(parents=True, exist_ok=True)
    plot_cells(rows, baseline, fig_dir / "fig_2x2.png", args.comm_range, args.k)
    plot_dose(rows, fig_dir / "fig_dose.png", args.comm_range, args.dose_cell)
    plot_fix(rows, baseline, fig_dir / "fig_fix.png", args.best_cell, args.k)
    plot_scale(rows, baseline, fig_dir / "fig_scale256.png", args.k)
    print("figures written to", fig_dir)


if __name__ == "__main__":
    main()

## Balayages

In [ ]:
# Balayage complet, reprenable. Le GPU sert au forward de LPAC ; les CVT tournent en C++ (CPU).
import subprocess, sys
cmd = [sys.executable, "-m", "lpac_study.sweep", "--data-dir", str(DATA_DIR), "--out-dir", str(OUT_DIR / "raw"),
       "--num-envs", str(NUM_ENVS), "--num-steps", str(NUM_STEPS), "--ranges", *map(str, RANGES),
       "--constant-degree", *CONSTANT_DEGREE, "--device", "auto"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)

## Résultats

In [ ]:
# Fusion (un seul processus ici), tableau récapitulatif et figures.
import subprocess, sys
subprocess.run([sys.executable, "-m", "lpac_study.merge", "--parts", str(OUT_DIR / "raw"), "--out-dir", str(OUT_DIR)], check=True)
subprocess.run([sys.executable, "-m", "lpac_study.plots", "--results-dir", str(OUT_DIR), "--fig-dir", str(OUT_DIR)], check=True)
from IPython.display import Image, display
for name in ["fig_portee.png", "fig_degre.png", "fig_degre_constant.png", "fig_rang.png"]:
    if (OUT_DIR / name).is_file():
        display(Image(filename=str(OUT_DIR / name)))

## Inference-time factorial: which input carries the neighbourhood that breaks the model?

The neighbourhood enters LPAC twice: as the `edge_index` of the five `TAGConv` layers (factor
**GRAPH**: G0 every edge in range, G1 the k nearest robots in range, made undirected with
`torch_geometric.utils.to_undirected`) and as the two communication-map channels of the CNN input
(factor **MAP**: C0 the library's map, C1 only the k nearest, C2 only robots within 128 m drawn at the
128 m resolution). The simulator always runs at the test range; only the model inputs change. The
checks cell first verifies that the rebuilt maps and graph match the library exactly.

In [ ]:
subprocess.run([sys.executable, "-m", "lpac_study.checks", "--data-dir", str(DATA_DIR), "--comm-range", "1024", "--steps", "10"], check=True)

In [ ]:
# Factorial at 1024 m (six cells, k = 4), normalized by the clairvoyant rows of the range sweep above.
FACT_RANGES, FACT_K, FACT_ENVS, FACT_STEPS = ["1024"], ["4"], NUM_ENVS, NUM_STEPS
if QUICK:
    FACT_ENVS, FACT_STEPS = 1, 20
cmd = [sys.executable, "-m", "lpac_study.factorial", "--data-dir", str(DATA_DIR), "--baseline", str(OUT_DIR / "results.csv"),
       "--out-dir", str(OUT_DIR / "factorial"), "--ranges", *FACT_RANGES, "--k", *FACT_K,
       "--num-envs", str(FACT_ENVS), "--num-steps", str(FACT_STEPS)]
subprocess.run(cmd, check=True)
subprocess.run([sys.executable, "-m", "lpac_study.plots_factorial", "--results", str(OUT_DIR / "factorial" / "results_2x2.csv"),
                "--baseline", str(OUT_DIR / "results.csv"), "--fig-dir", str(OUT_DIR)], check=True)
for name in ["fig_2x2.png", "fig_dose.png", "fig_fix.png"]:
    if (OUT_DIR / name).is_file():
        display(Image(filename=str(OUT_DIR / name)))

In [ ]:
# Archive à télécharger et à rapporter.
import shutil
zip_path = shutil.make_archive(str(OUT_DIR / "lpac_results"), "zip", root_dir=OUT_DIR,
                               base_dir=".", verbose=False)
print("archive :", zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except Exception as error:  # hors Colab
    print("téléchargement manuel :", error)